# 05b — Classification of one-road starting advantage
Run 05_starting_advantage first. This optional notebook predicts the theoretical benchmark, not actual winners. Blocked players are identified geometrically and excluded from the supervised fit. Training-only quartiles define Strong, Average and Weak; game-grouped validation prevents sharing boards across splits.

In [1]:
from pathlib import Path
import pandas as pd
from advantage_analysis import compare, FEATURES
from experiment_io import OUT, read_required, validate_advantage, same_players, validate_players
print("Working directory:",Path.cwd())
d=validate_advantage(read_required(OUT,"starting_advantage.csv","05_starting_advantage.ipynb"))
p=validate_players(read_required(OUT,"initial_placements.csv","01_data_extraction.ipynb"))
assert same_players(d,p), "Starting advantage is from a different dataset. Rerun 05a."
print(f"Games: {d.Game_ID.nunique():,}; players: {len(d):,}; eligible for classification: {(~d.Blocked).sum():,}")
display(d.Status.value_counts())

Working directory: d:\Projects\catanatron-513project
Games: 5,000; players: 20,000; eligible for classification: 19,160


Status
Reached           18741
Blocked             840
Beyond horizon      419
Name: count, dtype: int64

In [2]:
# Each classifier is evaluated on a grouped holdout and five grouped CV folds.
results, cv, predictions, labeled, q = compare(d)
assert q[1]<40, "Horizon must exceed the weak-class boundary"
print("Train-only quartile boundaries:",q)
print("Features:",FEATURES)
display(labeled.groupby(["Split","Advantage_Class"]).size())
display(results.round(4))
display(cv.round(4))
results.to_csv(OUT/"advantage_classification_results.csv",index=False)
cv.to_csv(OUT/"advantage_classification_cv.csv",index=False)
predictions.to_csv(OUT/"advantage_predictions.csv",index=False)
labeled.to_csv(OUT/"advantage_labeled_positions.csv",index=False)
print("Classification files saved to",OUT.resolve())

Training Majority baseline (holdout + five grouped folds)
Training KNN (holdout + five grouped folds)
Training Naive Bayes (holdout + five grouped folds)
Training CART (holdout + five grouped folds)
Training Random Forest (holdout + five grouped folds)
Training Logistic Regression (holdout + five grouped folds)
Train-only quartile boundaries: (8.0, 15.0)
Features: ['Player_Number', 'Wood_Production', 'Brick_Production', 'Sheep_Production', 'Wheat_Production', 'Ore_Production', 'Number_Diversity', 'Expansion_Options', 'Initial_Wood', 'Initial_Brick', 'Initial_Sheep', 'Initial_Wheat', 'Initial_Ore']


Split  Advantage_Class
Test   Average            1642
       Blocked             159
       Strong             1269
       Weak                930
Train  Average            6595
       Blocked             681
       Strong             5102
       Weak               3622
dtype: int64

,Model,Accuracy,Balanced_Accuracy,Macro_F1
0,Majority baseline,0.4275,0.3333,0.1996
1,KNN,0.8355,0.8358,0.8402
2,Naive Bayes,0.7592,0.7595,0.7663
3,CART,0.7618,0.7530,0.7663
4,Random Forest,0.8907,0.8888,0.8940
5,Logistic Regression,0.8573,0.8620,0.8639


,Model,Accuracy,Balanced_Accuracy,Macro_F1,Accuracy_SD,Balanced_Accuracy_SD,Macro_F1_SD
0,Majority baseline,0.4299,0.3333,0.2004,0.0060,0.0000,0.0020
1,KNN,0.8263,0.8279,0.8313,0.0051,0.0043,0.0049
2,Naive Bayes,0.7606,0.7628,0.7682,0.0075,0.0079,0.0075
3,CART,0.7669,0.7613,0.7719,0.0071,0.0075,0.0072
4,Random Forest,0.8870,0.8846,0.8901,0.0090,0.0085,0.0083
5,Logistic Regression,0.8564,0.8619,0.8636,0.0061,0.0054,0.0058


Classification files saved to D:\Projects\catanatron-513project\processed_data_5000


The outputs are newly calculated from the 5,000-game dataset. Use 06 for comparison charts and 07/08 for actual winner analysis.